# 05 - RQ3: Are cancellation rates associated with the operating profile of carrier-airport clusters?

* **Unit of analysis:** carrier x origin airport (one row per carrier at an airport, all analysed years pooled).
* **Clustering features (operational / capacity):** average load factor, mean monthly departures (log), average distance, variability of monthly departures across months (CV).
  Cancellation is **not** used to form clusters - it is the outcome tested afterwards (cluster-then-test design).
* **DV:** flight cancellation (binary at flight level; a rate at carrier-airport level).
* **H0:** cancellation rate is independent of cluster. **H1:** it depends on cluster.
* **Tests:** chi-square test of independence on all flights (+ Cramer's V); planned two-proportion z-test between the highest- and lowest-cancellation clusters;
  and, because flights within one carrier-airport are not independent, a Kruskal-Wallis test on the carrier-airport cancellation rates.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from labels import MONTH_NAMES, DOW_NAMES, carrier_name, airport_name
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from plot_style import set_style, save_fig
set_style()
ca = pd.read_csv(PROCESSED / "carrier_airport.csv")
feats = ["avg_load_factor", "mean_monthly_departures", "avg_distance", "seasonal_cv"]
Z = ca[feats].copy(); Z["mean_monthly_departures"] = np.log1p(Z["mean_monthly_departures"])   # departures are strongly right-skewed
Zs = StandardScaler().fit_transform(Z)
print(f"{len(ca):,} carrier-airport units, {int(ca['flights_scheduled'].sum()):,} flights (minimum required total: {load_results()['sample_size']['rq3_total']:,})")
ca[feats].describe().round(2)

1,598 carrier-airport units, 14,044,433 flights (minimum required total: 6,152)


,avg_load_factor,mean_monthly_departures,avg_distance,seasonal_cv
count,1598.00,1598.00,1598.00,1598.00
mean,78.65,370.78,761.93,0.32
std,8.54,1028.77,523.68,0.23
min,17.09,7.52,60.11,0.03
25%,75.95,52.71,443.19,0.15
50%,80.63,99.90,648.84,0.26
75%,83.69,254.83,927.58,0.44
max,93.77,19170.50,5095.00,1.72


## Choose k (elbow + silhouette)

In [2]:
rows, fits = [], {}
for k in range(2, 9):
    km = KMeans(k, n_init=20, random_state=RANDOM_STATE).fit(Zs)
    sizes = np.bincount(km.labels_)
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(Zs, km.labels_), "smallest_cluster": int(sizes.min())})
    fits[k] = km
sel = pd.DataFrame(rows)
# eligible: every cluster has at least 5% of the units; among those pick the best silhouette
elig = sel[sel["smallest_cluster"] >= max(5, 0.05 * len(ca))]
k_best = int(elig.loc[elig["silhouette"].idxmax(), "k"]) if len(elig) else int(sel.loc[sel["silhouette"].idxmax(), "k"])
sel.to_csv(TAB / "rq3_k_selection.csv", index=False)
print("chosen k =", k_best); sel.round(3)

chosen k = 4


,k,inertia,silhouette,smallest_cluster
0,2,4764.457,0.261,737
1,3,3834.319,0.290,101
2,4,2978.603,0.320,100
3,5,2402.903,0.289,97
4,6,2142.513,0.246,93
5,7,1914.188,0.254,42
6,8,1761.193,0.252,41


In [3]:
km = KMeans(k_best, n_init=50, random_state=RANDOM_STATE).fit(Zs)
# relabel clusters 1..k in order of increasing average load factor so the numbering is stable and meaningful
order = pd.Series(ca.groupby(km.labels_)["avg_load_factor"].mean()).sort_values().index
remap = {old: new for new, old in enumerate(order, start=1)}
ca["cluster"] = [remap[l] for l in km.labels_]
sil_final = float(silhouette_score(Zs, ca["cluster"]))

prof = (ca.groupby("cluster").agg(units=("cluster", "size"), flights=("flights_scheduled", "sum"), cancelled=("cancelled", "sum"),
                                   load_factor=("avg_load_factor", "mean"), monthly_departures=("mean_monthly_departures", "median"),
                                   distance=("avg_distance", "mean"), seasonal_cv=("seasonal_cv", "mean")))
prof["cancel_rate"] = prof["cancelled"] / prof["flights"] * 100
lo, hi = proportion_confint(prof["cancelled"], prof["flights"], method="wilson")
prof["ci_lo"], prof["ci_hi"] = lo * 100, hi * 100
top_car = ca.groupby("cluster")["carrier"].agg(lambda s: ", ".join(s.value_counts().head(3).index.astype(str)))
prof["top_carriers"] = top_car
prof.round(3).to_csv(TAB / "rq3_cluster_profile.csv")
prof.round(2)

,units,flights,cancelled,load_factor,monthly_departures,distance,seasonal_cv,cancel_rate,ci_lo,ci_hi,top_carriers
cluster,,,,,,,,,,,
1,106,163909,2946,55.31,52.50,385.00,0.25,1.80,1.73,1.86,"OO, AS, F9"
2,623,777800,15143,79.67,52.70,659.13,0.52,1.95,1.92,1.98,"MQ, G4, OO"
3,771,12812387,176537,80.30,252.67,698.79,0.18,1.38,1.37,1.38,"OO, DL, WN"
4,98,290337,3715,84.48,70.21,2319.85,0.25,1.28,1.24,1.32,"AS, HA, B6"


## Test: is cancellation rate associated with cluster?

In [4]:
# (a) flight-level chi-square test of independence (cluster x cancelled)
tab = np.column_stack([prof["cancelled"], prof["flights"] - prof["cancelled"]]).astype(int)
chi2, p_chi, dof, _ = stats.chi2_contingency(tab)
N = tab.sum(); cramers_v = float(np.sqrt(chi2 / (N * (min(tab.shape) - 1))))

# (b) planned contrast: highest- vs lowest-cancellation cluster
hi_c, lo_c = prof["cancel_rate"].idxmax(), prof["cancel_rate"].idxmin()
zstat, p_z = proportions_ztest([prof.loc[hi_c, "cancelled"], prof.loc[lo_c, "cancelled"]], [prof.loc[hi_c, "flights"], prof.loc[lo_c, "flights"]])

# (c) unit-level test (carrier-airports are the independent units; avoids pseudo-replication across flights)
groups = [g["cancel_rate"].values for _, g in ca.groupby("cluster")]
H, p_kw = stats.kruskal(*groups)
eps2 = float((H - k_best + 1) / (len(ca) - k_best))
print(f"chi2({dof}) = {chi2:,.1f}, p = {p_chi:.3g}, Cramer's V = {cramers_v:.4f}")
print(f"cluster {hi_c} vs {lo_c}: z = {zstat:.2f}, p = {p_z:.3g}")
print(f"Kruskal-Wallis on carrier-airport rates: H({k_best-1}) = {H:.2f}, p = {p_kw:.3g}, epsilon^2 = {eps2:.3f}")

chi2(3) = 1,917.0, p = 0, Cramer's V = 0.0117
cluster 2 vs 4: z = 23.30, p = 4.4e-120
Kruskal-Wallis on carrier-airport rates: H(3) = 40.94, p = 6.74e-09, epsilon^2 = 0.024


In [5]:
# --- figures ---------------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(6.5, 2.9))
ax[0].plot(sel["k"], sel["inertia"], marker="o", color=PALETTE[0]); ax[0].set_xlabel("Number of clusters (k)"); ax[0].set_ylabel("Within-cluster sum of squares")
ax[1].plot(sel["k"], sel["silhouette"], marker="o", color=PALETTE[1]); ax[1].set_xlabel("Number of clusters (k)"); ax[1].set_ylabel("Silhouette score")
ax[1].axvline(k_best, color=INK_2, ls="--", lw=1); ax[1].text(k_best + 0.1, ax[1].get_ylim()[0] + 0.02 * np.ptp(ax[1].get_ylim()), f"k = {k_best}", color=INK)
for a in ax: a.set_xticks(range(2, 9))
fig.tight_layout(); save_fig(fig, "fig17_rq3_k_selection")

pcs = PCA(2, random_state=RANDOM_STATE).fit(Zs); P = pcs.transform(Zs)
mk = ["o", "s", "^", "D", "v", "P"]
fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.3))
for c in range(1, k_best + 1):
    m = ca["cluster"].values == c
    ax[0].scatter(P[m, 0], P[m, 1], s=14, marker=mk[(c - 1) % len(mk)], color=PALETTE[c - 1], alpha=0.85, label=f"Cluster {c}", linewidths=0.4, edgecolors="white")
ax[0].set_xlabel(f"Principal component 1 ({pcs.explained_variance_ratio_[0]*100:.0f}% of variance)"); ax[0].set_ylabel(f"Principal component 2 ({pcs.explained_variance_ratio_[1]*100:.0f}%)")
ax[0].legend(fontsize=7, loc="best")
ax[1].bar(prof.index.astype(str), prof["cancel_rate"], color=[PALETTE[c - 1] for c in prof.index], width=0.65,
          yerr=[prof["cancel_rate"] - prof["ci_lo"], prof["ci_hi"] - prof["cancel_rate"]], capsize=3, ecolor=INK_2)
for i, (c, v) in enumerate(prof["cancel_rate"].items()): ax[1].text(i, prof.loc[c, "ci_hi"] + 0.03, f"{v:.2f}%", ha="center", fontsize=8, color=INK)
ax[1].set_xlabel("Cluster (ordered by load factor)"); ax[1].set_ylabel("Cancelled flights (% of scheduled,\n95% confidence interval)")
fig.tight_layout(); save_fig(fig, "fig18_rq3_clusters")

rq3 = {
    "units": int(len(ca)), "flights": int(ca["flights_scheduled"].sum()), "min_required_total": load_results()["sample_size"]["rq3_total"],
    "k": k_best, "silhouette_final": sil_final, "k_selection": sel.round(4).to_dict("records"),
    "pca_variance": [float(v) for v in pcs.explained_variance_ratio_],
    "profile": prof.round(4).reset_index().to_dict("records"),
    "chi2": float(chi2), "chi2_df": int(dof), "chi2_p": float(p_chi), "cramers_v": cramers_v,
    "contrast": {"high_cluster": int(hi_c), "low_cluster": int(lo_c), "z": float(zstat), "p": float(p_z),
                 "rate_high": float(prof.loc[hi_c, "cancel_rate"]), "rate_low": float(prof.loc[lo_c, "cancel_rate"])},
    "kruskal_H": float(H), "kruskal_p": float(p_kw), "kruskal_epsilon2": eps2,
    "overall_cancel_rate": float(ca["cancelled"].sum() / ca["flights_scheduled"].sum() * 100),
    "decision_flight_level": "reject H0" if p_chi < 0.05 else "fail to reject H0",
    "decision_unit_level": "reject H0" if p_kw < 0.05 else "fail to reject H0",
}
save_result("rq3", rq3)
print(rq3["decision_flight_level"], "(flight level) |", rq3["decision_unit_level"], "(unit level)")

reject H0 (flight level) | reject H0 (unit level)
